# SLM Pretraining trên TF1 — English Fable Generator

Train một **small language model (SLM) from scratch** trên bộ dữ liệu fable TF1 để chứng minh một model rất nhỏ (~30M tham số) vẫn viết được truyện ngụ ngôn thiếu nhi mạch lạc.

**Bản này là PHA 2** (2026-07-11): resume từ checkpoint pha 1 (loss 1.447) và train tiếp tới STEPS 3600 trên **corpus v2** với 2 can thiệp data rút từ đánh giá định tính: cap template "wise old owl" xuống 10% (pha 1 sinh owl trong 90% truyện) + giảm slot dropout teaching/outcome xuống 0.15 (model học bám moral/outcome được yêu cầu). Artifact pha 2 mang hậu tố `p2`, KHÔNG ghi đè kết quả pha 1.

**Cách chạy:** `Runtime -> Change runtime type -> T4 GPU`, rồi `Runtime -> Run all`.
Mọi artifact (checkpoint, model, GGUF, Modelfile, analysis) đều ghi vào **thư mục làm việc trên Google Drive** (tự dò theo `ckpt_30M`, nên rename/move thư mục vẫn chạy đúng) và sống sót khi runtime bị recycle.

**Pipeline:** mount Drive -> setup & data -> hyperparameters -> dataset (loss-masked) -> model + WSD training -> train 30M -> Step 5b analysis dashboard -> export sang Ollama.

## Resume sau khi disconnect (không mất công đoạn nào)

Colab free có thể ngắt runtime giữa chừng. Mọi bước tốn công đều được cache lên Drive nên không bao giờ phải làm lại từ đầu:

- **Corpus + tokenizer** cache ở `slm_tf1/data_tf1` (Step 1 khôi phục trong vài giây thay vì tải lại).
- **Checkpoint training** ghi vào `slm_tf1/ckpt_30M` mỗi 500 step; `train("30M")` **tự resume** từ checkpoint mới nhất.
- **Model cuối** lưu ở `slm_tf1/30M`; harness phân tích tự load từ đó nếu bản local đã mất.

**Cách resume sau recycle:** reconnect, chọn T4 GPU, mount Drive (cell 1), rồi `Runtime -> Run all`. Step 1 khôi phục corpus từ cache, training chạy tiếp từ checkpoint cuối. Nếu model đã train xong (tồn tại thư mục `slm_tf1/30M`), có thể nhảy thẳng tới các cell Step 5b.

In [ ]:
# --- Mount Google Drive + định vị thư mục làm việc (chứa ckpt_30M của pha 1) ---
from google.colab import drive
drive.mount('/content/drive')                 # bấm "Authorize" trong popup
import os
from collections import deque

def _find_drive_root():
    # thư mục làm việc có thể đã được di chuyển trong Drive -> tự dò theo ckpt_30M
    base = "/content/drive/MyDrive"
    q = deque([(base, 0)])
    while q:
        d, depth = q.popleft()
        try:
            entries = [e for e in os.scandir(d) if e.is_dir()]
        except (PermissionError, FileNotFoundError):
            continue
        for e in entries:
            if e.name == "ckpt_30M":
                return d
            if depth < 3 and not e.name.startswith("."):
                q.append((e.path, depth + 1))
    return None

DRIVE = _find_drive_root() or "/content/drive/MyDrive/slm_tf1"
os.makedirs(DRIVE, exist_ok=True)
print("DRIVE =", DRIVE)

## Step 1 - Setup & data

Clone repo, cài dependency, rồi build corpus training + tokenizer (idempotent - bỏ qua nếu đã có, khôi phục từ cache Drive sau recycle).

- **Corpus:** ~**400k** fable TF1 (lớn hơn baseline 150k), lọc chất lượng 60-320 từ. Mỗi example có format `conditioning (5 narrative slot) \n <|story|> fable <|end|>`.
- **Tokenizer:** **BPE (vocab 12k)** tự train trên chính corpus fable - giữ embedding table của model nhỏ gọn.

In [ ]:
# --- Clone repo + cài dependency + build (hoặc khôi phục) corpus V2 ---
import os, json, subprocess, shutil
if not os.path.exists("/content/tinystory-vn"):
    subprocess.run("git clone -q https://github.com/tungd/tinystory-vn.git", shell=True, cwd="/content")
os.chdir("/content/tinystory-vn")
subprocess.run("git fetch -q origin && git checkout -q feat/slm-pretrain-tf1 && git pull -q", shell=True)
subprocess.run('pip -q install "datasets>=2.20" "tokenizers>=0.19" "transformers>=4.44" torch accelerate', shell=True)

TRAIN_N       = 400_000
DATA_CACHE    = f"{DRIVE}/data_tf1"       # corpus pha 1 (chỉ dùng để lấy lại tokenizer)
DATA_CACHE_V2 = f"{DRIVE}/data_tf1_v2"    # corpus pha 2 (2 can thiệp data bên dưới)
os.makedirs("data", exist_ok=True)
if os.path.exists(f"{DATA_CACHE_V2}/tokenizer.json"):
    print("khôi phục corpus v2 từ cache Drive (không build lại)...")
    shutil.rmtree("data/tf1", ignore_errors=True)
    shutil.copytree(DATA_CACHE_V2, "data/tf1")
else:
    # 2 can thiệp data (từ đánh giá định tính 2026-07-11):
    #   - cap "wise old owl": có trong 28% fable thật nhưng model khuếch đại lên 90%
    #     khi sinh (mode amplification) -> giới hạn 10% corpus để giảm template collapse.
    #   - slot dropout teaching/outcome 0.3 -> 0.15: model thấy moral/outcome trong
    #     conditioning ~85% thay vì 67% -> học BÁM moral được yêu cầu thay vì tự bịa.
    # Tokenizer GIỮ NGUYÊN từ pha 1 (đổi tokenizer sẽ vô hiệu checkpoint resume).
    subprocess.run(f"python -m trieulh.scripts.prepare_tf1_pretrain --train-n {TRAIN_N} --test-n 500 "
                   f"--min-words 60 --max-words 320 "
                   f'--cap-phrase "wise old owl" --cap-frac 0.10 '
                   f"--slot-dropout teaching=0.15 outcome=0.15 --out data/tf1", shell=True)
    shutil.copy(f"{DATA_CACHE}/tokenizer.json", "data/tf1/tokenizer.json")
    shutil.copytree("data/tf1", DATA_CACHE_V2, dirs_exist_ok=True)
    print("đã build corpus v2 và cache lên Drive")

# kiểm chứng can thiệp data ngay tại chỗ
_texts = [json.loads(l)["text"].lower() for l in open("data/tf1/train.jsonl")]
_owl = sum(1 for t in _texts if "wise old owl" in t); _n = len(_texts)
print(f"data sẵn sàng: {_n} dòng | tỷ lệ owl {_owl/_n:.1%} (mục tiêu ~10%)")
del _texts

## Step 2 - Training hyperparameters

**"Phương pháp" của quá trình train** - mọi knob có thể chỉnh: nó làm gì, khoảng an toàn, và hệ quả khi lệch khỏi khoảng đó. Sửa ở đây; các cell bên dưới đọc các giá trị này.

In [ ]:
# =========================== TRAINING HYPERPARAMETERS ===========================
# Mỗi knob: nó làm gì, khoảng an toàn cho CHÍNH setup này (30M tham số, vocab 12k,
# fable TF1, T4 16GB), và hệ quả nếu ra ngoài khoảng đó.

SEQ_LEN = 512            # số token tối đa mỗi example. Fable gói gọn trong ~400 token nên 512
                         #   chứa đủ prompt + truyện. Lớn hơn -> chi phí attention tăng bậc hai
                         #   mà không được gì; nhỏ hơn (256) -> fable dài bị cắt cụt và model
                         #   không bao giờ học được cách kết truyện.

# Kiến trúc model (Llama-style). v2: chỉ train 30M.
ARCH = {
    "30M": dict(
        hidden_size=512,          # độ rộng embedding/residual stream; chiếm phần lớn số tham số.
        intermediate_size=2048,   # độ rộng FFN, chuẩn 4x hidden. Nhỏ hơn -> ít capacity hơn.
        num_hidden_layers=8,      # độ sâu = khả năng composition. <6 làm giảm độ mạch lạc của
                                  #   truyện; sâu hơn chỉ đáng khi có nhiều token training hơn
                                  #   (scaling laws).
        num_attention_heads=8,    # 512/8 = 64 dim mỗi head, điểm ngọt tiêu chuẩn.
        num_key_value_heads=2,    # GQA: 4 query head dùng chung 1 kv head -> tiết kiệm KV memory,
                                  #   gần như không mất chất lượng ở scale này.
    ),
}

# Optimizer (AdamW)
PEAK_LR      = 3e-3        # learning rate đỉnh. Model nhỏ train from scratch cần LR cao.
                           #   >5e-3: nguy cơ loss spike / diverge. <1e-3: hội tụ quá chậm,
                           #   phí phút GPU. Khỏe mạnh = giảm mượt, đơn điệu như lần chạy đã
                           #   quan sát (7.2 -> ~1.5).
ADAM_BETAS   = (0.9, 0.95) # beta2 = 0.95 (không phải 0.999) thích ứng nhanh hơn với biến động
                           #   variance của gradient; lựa chọn chuẩn cho LM pretraining.
WEIGHT_DECAY = 0.1         # L2 regularization. 0.05-0.1 là chuẩn; 0 dễ overfit trên data lặp,
                           #   >0.3 underfit (trọng số bị kéo về 0).
GRAD_CLIP    = 1.0         # chặn trần gradient norm để một batch xấu không gây loss spike.
                           #   Chỉ tăng nếu gradient bị clip liên tục (việc học bị chậm lại).

# Lịch learning rate: Warmup-Stable-Decay (WSD)
WARMUP_FRAC = 0.02         # 2% số step đầu tăng LR 0 -> peak. <1% dễ loss spike sớm ở LR cao;
                           #   >10% phí step ở LR thấp.
DECAY_FRAC  = 0.20         # 20% số step cuối giảm LR peak -> 0; pha này tạo cú giảm loss chốt.
                           #   <10% bỏ lỡ phần loss cuối; >40% rút ngắn pha stable vốn đảm
                           #   nhiệm phần lớn việc học.

# Batch & số step
BATCH_SIZE   = 32          # số sequence mỗi forward/backward; giới hạn bởi VRAM T4 tại
                           #   SEQ_LEN=512 + fp16 (lớn hơn dễ OOM).
GRAD_ACCUM   = 4           # tích lũy 4 micro-batch -> effective batch 128 sequence (~33k token
                           #   truyện mỗi update). Accum cao = gradient mượt hơn nhưng ít
                           #   update hơn mỗi phút.
STEPS        = 3600        # PHA 2: resume từ checkpoint pha 1 (step 1500) và chạy tiếp tới
                           #   3600 trên corpus v2 -> thêm ~2100 step (~55-60 phút trên T4).
                           #   Đây là kiểu "mid-training annealing trên data sạch" (MiniCPM,
                           #   Llama-3): pha sau học phân bố đã sửa, chủ động kéo model khỏi
                           #   owl-prior. Loss kỳ vọng 1.447 -> ~1.36-1.40. Chinchilla-optimal
                           #   cho 30M (~600M token) là ~7900 step nếu còn quota.
FP16         = True        # mixed precision: nhanh ~2x, tốn nửa VRAM trên T4. Nếu loss thành
                           #   NaN (hiếm ở scale này), chuyển sang fp32.
LOG_EVERY    = 25          # log loss mỗi N step; nguồn dữ liệu cho các curve. Nhỏ hơn = đồ thị
                           #   dày điểm hơn nhưng nhiễu hơn.
print("hyperparameters set. size: 30M | steps:", STEPS, "| eff. batch:", BATCH_SIZE*GRAD_ACCUM)

## Step 3 - Dataset với loss masking

Tokenize từng example và **mask phần conditioning prefix** (labels = -100) để model chỉ học predict **truyện**, không học predict prompt. `cond_len` (offset theo ký tự) được đổi sang số **token** trước khi mask. Collator pad theo batch, đánh dấu padding bằng label -100 / attention_mask 0.

In [ ]:
# --- Build dataset đã tokenize + loss-mask, kèm padding collator ---
import json, torch
from transformers import (LlamaConfig, LlamaForCausalLM, PreTrainedTokenizerFast,
                          Trainer, TrainingArguments)

tok = PreTrainedTokenizerFast(tokenizer_file="data/tf1/tokenizer.json",
        unk_token="<|unk|>", pad_token="<|pad|>", eos_token="<|end|>")

def encode(row):
    ids = tok(row["text"], truncation=True, max_length=SEQ_LEN)["input_ids"]
    n_cond = min(len(tok(row["text"][:row["cond_len"]])["input_ids"]), len(ids))
    labels = [-100] * n_cond + ids[n_cond:]     # mask token prompt; chỉ học phần truyện
    return {"input_ids": ids, "labels": labels}

print("encoding..."); DS = [encode(json.loads(l)) for l in open("data/tf1/train.jsonl")]
print("encoded", len(DS), "examples")

def collator(features):
    pad = tok.pad_token_id
    m = max(len(f["input_ids"]) for f in features)
    fill = lambda seq, val: seq + [val] * (m - len(seq))
    return {
        "input_ids":      torch.tensor([fill(f["input_ids"], pad)  for f in features]),
        "labels":         torch.tensor([fill(f["labels"], -100)    for f in features]),
        "attention_mask": torch.tensor([[1]*len(f["input_ids"]) + [0]*(m-len(f["input_ids"])) for f in features]),
    }

## Step 4 - Model + WSD training loop

`train(size)` dựng **model Llama-style từ random init** (`tie_word_embeddings=True` chia sẻ input/output embedding) và train bằng **AdamW** + lịch LR **Warmup-Stable-Decay**. Checkpoint ghi lên Drive mỗi 500 step và run **tự resume** từ checkpoint mới nhất sau recycle. Mọi giá trị đọc từ cell hyperparameters.

In [ ]:
# --- Train PHA 2: resume từ checkpoint pha 1, artifact hậu tố -p2 (không đè Run 3) ---
import os, glob
def train(size):
    tag = f"{size}-p2"
    ckpt_dir = f"{DRIVE}/ckpt_{size}_p2"            # checkpoint pha 2 (Drive, resumable)
    # ưu tiên checkpoint pha 2 (nếu pha 2 từng bị cắt); fallback checkpoint pha 1
    p2 = sorted(glob.glob(f"{ckpt_dir}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
    p1 = sorted(glob.glob(f"{DRIVE}/ckpt_{size}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
    resume_from = p2[-1] if p2 else (p1[-1] if p1 else None)
    assert resume_from, "không có checkpoint pha 1 để resume - chạy pha 1 trước"
    print(f"[{tag}] resume từ:", resume_from)

    model = LlamaForCausalLM.from_pretrained(resume_from)   # nạp weights pha 1
    print(f"[{tag}] params(M):", round(sum(p.numel() for p in model.parameters())/1e6, 1))

    optimizer = torch.optim.AdamW(model.parameters(), lr=PEAK_LR,
                                  betas=ADAM_BETAS, weight_decay=WEIGHT_DECAY)

    def wsd(step):                                  # hệ số nhân Warmup-Stable-Decay trong [0,1]
        warm, dec = int(WARMUP_FRAC*STEPS), int(DECAY_FRAC*STEPS)
        if step < warm:          return step / max(1, warm)               # 0 -> 1  (warmup)
        if step > STEPS - dec:   return max(0.0, (STEPS-step)/max(1,dec))  # 1 -> 0  (decay)
        return 1.0                                                         # giữ peak (stable)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, wsd)
    # LƯU Ý: checkpoint 1500 nằm trong pha stable của lịch mới (decay bắt đầu ở 2880)
    # -> LR quay về peak rồi decay lại cuối run, đúng thiết kế của WSD cho continued training.

    args = TrainingArguments(
        output_dir=ckpt_dir, max_steps=STEPS, fp16=FP16,
        per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
        max_grad_norm=GRAD_CLIP, logging_steps=LOG_EVERY,
        save_strategy="steps", save_steps=500, save_total_limit=2,
        lr_scheduler_type="constant", report_to=[],
        ignore_data_skip=True,   # corpus MỚI: học từ đầu data v2, không skip 192k example
    )
    trainer = Trainer(model=model, args=args, train_dataset=DS, data_collator=collator,
                      optimizers=(optimizer, scheduler))
    trainer.train(resume_from_checkpoint=resume_from)

    model.save_pretrained(f"out/{tag}"); tok.save_pretrained(f"out/{tag}")
    model.save_pretrained(f"{DRIVE}/{tag}"); tok.save_pretrained(f"{DRIVE}/{tag}")
    for d in (f"out/{tag}", f"{DRIVE}/{tag}"):
        p = f"{d}/tokenizer_config.json"; c = json.load(open(p))
        c["tokenizer_class"] = "PreTrainedTokenizerFast"; json.dump(c, open(p, "w"))
    import json as _J
    _J.dump(trainer.state.log_history, open(f"{DRIVE}/loss_log_{size}_p2.json", "w"))
    print(f"[{tag}] đã lưu -> out/{tag} và {DRIVE}/{tag}")
    return trainer          # <- để cell phân tích đọc được trainer.state.log_history

## Step 5 - Train model 30M (PHA 2: resume trên corpus v2)

Pha 2 nạp **checkpoint pha 1** (step 1500, loss ~1.61) và train tiếp tới **STEPS 3600** trên corpus v2 đã can thiệp. Loss mở đầu quanh ~1.6 (có thể nhích nhẹ vài chục step đầu do phân bố data đổi - bình thường), sau đó giảm tiếp; kỳ vọng kết thúc ~1.36-1.40. Khoảng **55-60 phút trên T4**; checkpoint ghi `ckpt_30M_p2` mỗi 500 step nên disconnect vẫn resume được. `train(...)` trả về `Trainer` để cell kế vẽ loss curve.

In [ ]:
trainer30 = train("30M")   # PHA 2: resume 1500 -> 3600 (~55-60 phút trên T4); checkpoint Drive mỗi 500 step

## Step 5b - Application analysis dashboard

Sau khi train, `collect_analysis()` gom mọi số liệu vào một dict (đồng thời dump lên Drive để sống sót khi runtime recycle). Ba figure render inline, và cell cuối in bảng verdict tự động.

**Cách đọc các figure** (ngưỡng là heuristic cho CHÍNH setup này: 30M tham số, vocab 12k, fable TF1):

- **Figure 1 - Training dynamics.** Loss phải giảm mượt qua các dải màu: trên 2.0 = undertrained, 1.5-2.0 = partially trained, dưới 1.5 = vùng target. Panel log-log kiểm chứng dự đoán của scaling law (Kaplan et al. 2020): loss theo step phải gần đường thẳng (R^2 > 0.95) sau warmup; cong rõ nghĩa là run đã lệch khỏi power-law regime (LR quá cao/thấp, hoặc data có vấn đề).
- **Figure 2 - Intrinsic quality vs fable thật.** Mỗi metric đặt model cạnh fable thật held-out. Distinct-1/2 (độ đa dạng) nên nằm trong ~15% giá trị thật; Self-BLEU (độ lặp, thấp = đa dạng hơn) trong 0.05 tuyệt đối; Flesch reading ease trong dải 80-100 đặc trưng của truyện thiếu nhi; histogram độ dài nên chồng lấp phân bố thật (>= 50%).
- **Figure 3 - Language-model behavior.** Perplexity held-out được neo giữa ceiling (vocab size 12000 = model đoán mò uniform) và floor (e^final-train-loss = mức mà train loss hứa hẹn). Tốt = sát floor, thấp hơn ceiling nhiều bậc. Mean loss theo vị trí tương đối trong truyện cho biết model tự tin ở đâu (mở truyện thường dễ nhất). Đường Zipf của text sinh ra nên bám sát đường thật.
- **Bảng verdict.** Mỗi metric kèm giá trị, target và PASS / WARN / FAIL, merge vào `analysis_30M.json` cho báo cáo.

Metric dùng lại `app/metrics.py` và `app/perplexity.py` từ repo. Biểu đồ render ngay trong run; không gọi judge ngoài ở đây.

In [ ]:
# --- Harness: gom mọi số liệu phân tích vào một dict, dump lên Drive ---
import os, sys, json, subprocess, torch
from collections import Counter
sys.path.insert(0, "/content/tinystory-vn")
subprocess.run("pip -q install textstat", shell=True)      # dependency cho metric readability
from app.metrics import distinct_n, self_bleu, flesch_reading_ease
from app.perplexity import aggregate_nll, perplexity_from_nll
from transformers import AutoModelForCausalLM

SEP, END = "<|story|>", "<|end|>"

# --- Các knob phân tích: ý nghĩa và tác động lên số liệu -------------------------
N_GEN  = 30    # số truyện sinh ra cho quality metric. 30 đủ thấy xu hướng
               #   (Distinct/Self-BLEU ổn định trong vài %); tăng lên 100+ để có
               #   error bar chuẩn báo cáo, đổi lại ~3x thời gian chạy.
N_PPL  = 200   # số doc held-out cho perplexity. 200 cho ước lượng ổn định (~+-2%);
               #   nhiều hơn chủ yếu tốn thời gian.
GEN_MAXNEW = 320   # trần độ dài sinh, khớp filter data 320 từ. Nhỏ hơn sẽ cắt cụt
                   #   truyện và làm lệch histogram độ dài.
GEN_TEMP   = 0.8   # sampling temperature, khớp default của app. >1.0 -> đa dạng hơn
                   #   nhưng nhiều lỗi ngữ pháp; <0.5 -> text lặp lại, Self-BLEU tăng.
POS_BINS = 16      # số bin cho profile loss theo vị trí tương đối (0-100% truyện).
# repetition_penalty 1.1 (không phải 1.3): penalty cao phạt việc lặp token tên nhân vật
# -> SLM nhỏ bị ép đổi nhân vật giữa truyện (entity drift, phát hiện khi review 2026-07-11).
V1_LOSS, V1_PPL = 1.8, None   # mốc baseline v1 (v1 không có số perplexity sạch)

def _story_of(text, cond_len):
    return text[cond_len:].replace(SEP, "").replace(END, "").strip()

def collect_analysis(size="30M"):
    A = {"size": size, "N_GEN": N_GEN, "N_PPL": N_PPL, "v1_loss": V1_LOSS, "v1_ppl": V1_PPL}

    # A. training dynamics (từ trainer30, hoặc loss log trên Drive nếu kernel đã bị recycle)
    try:
        hist = trainer30.state.log_history
    except NameError:
        hist = json.load(open(f"{DRIVE}/loss_log_{size.replace('-', '_')}.json"))
    A["steps"]  = [h["step"] for h in hist if "loss" in h]
    A["losses"] = [h["loss"] for h in hist if "loss" in h]
    A["lrs"]    = [h.get("learning_rate") for h in hist if "loss" in h]
    A["throughput"]  = hist[-1].get("train_samples_per_second")
    A["runtime_min"] = hist[-1].get("train_runtime", 0) / 60
    A["final_loss"] = A["losses"][-1] if A["losses"] else None
    A["vocab_size"] = tok.vocab_size

    # load model đã train: ưu tiên out/ local, fallback bản Drive (sống sót recycle)
    src = f"out/{size}" if os.path.isdir(f"out/{size}") else f"{DRIVE}/{size}"
    print(f"load model từ: {src}")
    model = AutoModelForCausalLM.from_pretrained(src).to("cuda").eval()
    tests = [json.loads(l) for l in open("data/tf1/test.jsonl")]

    # B. sinh fable + intrinsic quality so với fable thật held-out
    eos = tok.convert_tokens_to_ids(END)
    gen_stories = []
    for t in tests[:N_GEN]:
        cond = t["text"][:t["cond_len"]].rstrip("\n")
        ids = tok(cond + "\n" + SEP, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = model.generate(**ids, max_new_tokens=GEN_MAXNEW, do_sample=True,
                                  temperature=GEN_TEMP, top_p=0.9, repetition_penalty=1.1,
                                  eos_token_id=eos, pad_token_id=tok.pad_token_id)
        txt = tok.decode(out[0], skip_special_tokens=False)
        gen_stories.append(txt.split(SEP, 1)[1].replace(END, "").strip() if SEP in txt else txt)
    real_stories = [_story_of(t["text"], t["cond_len"]) for t in tests[:N_GEN]]
    # lưu TOÀN BỘ truyện sinh ra + conditioning (phục vụ báo cáo, tái kiểm metric)
    A["gen_stories"] = [{"cond": tests[i]["text"][:tests[i]["cond_len"]].strip(),
                         "story": gen_stories[i]} for i in range(len(gen_stories))]
    A["samples"] = A["gen_stories"][:3]

    def _flesch(texts):
        vals = [flesch_reading_ease(x) for x in texts if x.strip()]
        return sum(vals) / len(vals) if vals else 0.0
    A["quality"] = {
        "distinct1": {"gen": distinct_n(gen_stories, 1), "real": distinct_n(real_stories, 1)},
        "distinct2": {"gen": distinct_n(gen_stories, 2), "real": distinct_n(real_stories, 2)},
        "self_bleu": {"gen": self_bleu(gen_stories, 4),  "real": self_bleu(real_stories, 4)},
        "flesch":    {"gen": _flesch(gen_stories),        "real": _flesch(real_stories)},
    }
    A["len_gen"]  = [len(s.split()) for s in gen_stories]
    A["len_real"] = [len(s.split()) for s in real_stories]

    # C. LM behavior: perplexity held-out + mean loss theo vị trí TƯƠNG ĐỐI trong truyện
    per_seq = []
    pos_sum, pos_cnt = [0.0] * POS_BINS, [0] * POS_BINS
    lossf = torch.nn.CrossEntropyLoss(reduction="none", ignore_index=-100)
    for t in tests[:N_PPL]:
        enc = encode(t)                                   # dùng lại encoder của training (mask conditioning)
        ids = torch.tensor([enc["input_ids"]]).to("cuda")
        lab = torch.tensor([enc["labels"]]).to("cuda")
        with torch.no_grad():
            logits = model(ids).logits
        sl = logits[:, :-1].contiguous().view(-1, logits.size(-1))
        tl = lab[:, 1:].contiguous().view(-1)
        tok_loss = lossf(sl, tl)
        mask = tl != -100
        n_story = int(mask.sum())
        if n_story:
            per_seq.append((float(tok_loss[mask].sum()), n_story))
            story_losses = tok_loss[mask].tolist()        # loss của token truyện theo thứ tự
            for k, v in enumerate(story_losses):          # bin theo vị trí tương đối 0-100%
                b = min(POS_BINS - 1, int(k / max(1, n_story) * POS_BINS))
                pos_sum[b] += v; pos_cnt[b] += 1
    A["perplexity"] = perplexity_from_nll(aggregate_nll(per_seq), sum(n for _, n in per_seq))
    A["pos_loss"] = [pos_sum[i] / pos_cnt[i] if pos_cnt[i] else None for i in range(POS_BINS)]

    # Zipf: tần suất token (giảm dần) của output sinh ra vs vocabulary thật
    def _counts(texts):
        c = Counter()
        for s in texts: c.update(s.split())
        return sorted(c.values(), reverse=True)[:2000]
    A["zipf_gen"], A["zipf_real"] = _counts(gen_stories), _counts(real_stories)

    # tỷ lệ template owl trong output (đo hiệu quả của can thiệp cap-phrase pha 2)
    A["owl_rate_gen"] = sum(1 for s in gen_stories if "wise old owl" in s.lower()) / max(1, len(gen_stories))

    json.dump(A, open(f"{DRIVE}/analysis_{size.replace('-', '_')}.json", "w"))
    print(f"analysis xong | perplexity={A['perplexity']:.2f} | "
          f"owl-rate gen={A['owl_rate_gen']:.0%} (pha 1: 90%) | "
          f"distinct2 gen={A['quality']['distinct2']['gen']:.3f} real={A['quality']['distinct2']['real']:.3f}")
    for i, s in enumerate(A["samples"], 1):
        print(f"\n===== SAMPLE {i} =====\nCONDITIONING:\n{s['cond']}\nSTORY:\n{s['story'][:600]}")
    return A

analysis = collect_analysis("30M-p2")

In [ ]:
# --- Figure 1: training dynamics (loss kèm dải diễn giải, kiểm chứng power-law, LR, summary) ---
import numpy as np
import matplotlib.pyplot as plt

def fig_training(A):
    steps, losses = A.get("steps"), A.get("losses")
    if not steps:
        print("không có training log"); return
    fig, ax = plt.subplots(2, 2, figsize=(13, 8))

    # (1) loss với dải diễn giải (heuristic cho 30M / vocab 12k / TF1)
    a = ax[0, 0]
    lo, hi = min(1.0, min(losses) - 0.1), max(losses) + 0.3
    a.axhspan(2.0, hi, color="#fecaca", alpha=.35)     # > 2.0: undertrained
    a.axhspan(1.5, 2.0, color="#fef3c7", alpha=.45)    # 1.5-2.0: partially trained
    a.axhspan(lo, 1.5, color="#bbf7d0", alpha=.35)     # < 1.5: vùng target
    a.plot(steps, losses, color="#2563eb")
    a.axhline(A["v1_loss"], ls="--", color="#6b7280", label="v1 baseline ~1.8")
    a.axhline(1.447, ls=":", color="#0891b2", label="pha 1 final 1.447")
    a.set_ylim(lo, hi)
    a.set_title("Training loss (red undertrained / yellow partial / green target)")
    a.set_xlabel("step"); a.set_ylabel("cross-entropy"); a.legend(); a.grid(alpha=.3)

    # (2) kiểm chứng scaling law: log-log loss theo step phải ~tuyến tính sau warmup
    a = ax[0, 1]
    n0 = max(1, len(steps) // 10)                  # bỏ ~10% điểm đầu (giai đoạn warmup)
    xs, ys = np.log(steps[n0:]), np.log(losses[n0:])
    slope, intercept = np.polyfit(xs, ys, 1)
    pred = slope * xs + intercept
    r2 = float(1 - ((ys - pred) ** 2).sum() / max(1e-12, ((ys - ys.mean()) ** 2).sum()))
    a.loglog(steps, losses, color="#2563eb", label="loss")
    a.loglog(np.exp(xs), np.exp(pred), ls="--", color="#dc2626",
             label=f"fit: loss ~ step^{slope:.2f} (R^2 = {r2:.3f})")
    a.set_title("Scaling-law check (straight line on log-log = power law)")
    a.set_xlabel("step (log)"); a.set_ylabel("loss (log)")
    a.legend(); a.grid(alpha=.3, which="both")

    # (3) lịch learning rate (phải ra đúng hình WSD: tăng, đi ngang, giảm)
    a = ax[1, 0]
    lrs = A.get("lrs")
    if lrs and any(v is not None for v in lrs):
        a.plot(steps, lrs, color="#d97706")
        a.set_title("Learning rate (WSD schedule)")
        a.set_xlabel("step"); a.set_ylabel("lr"); a.grid(alpha=.3)
    else:
        a.text(.5, .5, "no LR logged", ha="center"); a.axis("off")

    # (4) tóm tắt run
    a = ax[1, 1]; a.axis("off")
    tps = A.get("throughput")
    txt = f"final loss: {losses[-1]:.3f} (v1 was ~{A['v1_loss']})\n"
    txt += f"total drop: {losses[0]:.2f} -> {losses[-1]:.3f}\n"
    txt += f"power-law exponent: {slope:.2f}, R^2 {r2:.3f}\n"
    txt += f"throughput: {tps:.1f} samples/sec\n" if tps else ""
    txt += f"runtime: {A.get('runtime_min', 0):.1f} min"
    a.text(.05, .85, txt, fontsize=13, family="monospace", va="top")
    a.set_title("Run summary")

    plt.tight_layout()
    plt.savefig(f"{DRIVE}/fig_training_{A['size']}.png", dpi=110, bbox_inches="tight")
    plt.show()

fig_training(analysis)

In [ ]:
# --- Figure 2: intrinsic quality so với fable thật held-out, kèm ngưỡng pass ---
import numpy as np
import matplotlib.pyplot as plt

GAP_PASS = 0.15          # Distinct-1/2 nằm trong 15% giá trị thật = PASS
SB_PASS  = 0.05          # Self-BLEU lệch tối đa 0.05 TUYỆT ĐỐI so với thật = PASS (giá trị
                         #   thật rất nhỏ nên gap tương đối sẽ nổ vì nhiễu)
FLESCH_BAND = (80, 100)  # dải Flesch reading-ease đặc trưng của truyện thiếu nhi

def fig_quality(A):
    q = A.get("quality")
    if not q:
        print("không có quality metric"); return
    fig, ax = plt.subplots(2, 2, figsize=(13, 8.5))

    # (1) Distinct-1/2 gộp nhóm, annotate gap tương đối so với thật
    a = ax[0, 0]
    d1g, d1r = q["distinct1"]["gen"], q["distinct1"]["real"]
    d2g, d2r = q["distinct2"]["gen"], q["distinct2"]["real"]
    g1 = abs(d1g - d1r) / max(1e-9, d1r); g2 = abs(d2g - d2r) / max(1e-9, d2r)
    pos, vals = [0, 1, 2.5, 3.5], [d1g, d1r, d2g, d2r]
    a.bar(pos, vals, color=["#2563eb", "#94a3b8", "#2563eb", "#94a3b8"])
    a.set_xticks(pos); a.set_xticklabels(["SLM d1", "real d1", "SLM d2", "real d2"])
    for p, v in zip(pos, vals): a.text(p, v, f"{v:.3f}", ha="center", va="bottom")
    ok1, ok2 = g1 <= GAP_PASS, g2 <= GAP_PASS
    a.set_title(f"Distinct-1/2 diversity. gap d1 {g1:.0%} ({'PASS' if ok1 else 'CHECK'}), "
                f"d2 {g2:.0%} ({'PASS' if ok2 else 'CHECK'}); <= {GAP_PASS:.0%} = PASS")
    a.grid(axis="y", alpha=.3)

    # (2) Self-BLEU: thấp = đa dạng hơn; đánh giá theo gap tuyệt đối
    a = ax[0, 1]
    sbg, sbr = q["self_bleu"]["gen"], q["self_bleu"]["real"]
    gsb = abs(sbg - sbr)
    a.bar(["SLM 30M", "real fables"], [sbg, sbr], color=["#2563eb", "#94a3b8"])
    for i, v in enumerate([sbg, sbr]): a.text(i, v, f"{v:.3f}", ha="center", va="bottom")
    a.set_title(f"Self-BLEU (lower = more diverse). abs gap {gsb:.3f} "
                f"({'PASS' if gsb <= SB_PASS else 'CHECK'}; <= {SB_PASS} = PASS)")
    a.grid(axis="y", alpha=.3)

    # (3) Flesch reading ease so với dải target của truyện thiếu nhi
    a = ax[1, 0]
    fg, fr = q["flesch"]["gen"], q["flesch"]["real"]
    a.axhspan(*FLESCH_BAND, color="#bbf7d0", alpha=.4,
              label=f"target {FLESCH_BAND[0]}-{FLESCH_BAND[1]} (children)")
    a.bar(["SLM 30M", "real fables"], [fg, fr], color=["#2563eb", "#94a3b8"])
    for i, v in enumerate([fg, fr]): a.text(i, v, f"{v:.1f}", ha="center", va="bottom")
    a.set_title("Flesch reading ease (80-100 = easy, right for children)")
    a.legend(); a.grid(axis="y", alpha=.3)

    # (4) phân bố độ dài truyện + hệ số chồng lấp (tổng min của 2 histogram chuẩn hóa)
    a = ax[1, 1]
    lg, lr = A["len_gen"], A["len_real"]
    bins = np.linspace(min(lg + lr), max(lg + lr), 16)
    hg, _ = np.histogram(lg, bins=bins); hr, _ = np.histogram(lr, bins=bins)
    overlap = float(np.minimum(hg / max(1, hg.sum()), hr / max(1, hr.sum())).sum())
    a.hist(lg, bins=bins, alpha=.6, label="SLM 30M", color="#2563eb")
    a.hist(lr, bins=bins, alpha=.6, label="real fables", color="#94a3b8")
    a.set_title(f"Story length (words). overlap {overlap:.0%} "
                f"({'PASS' if overlap >= 0.5 else 'CHECK'}; >= 50% = PASS)")
    a.set_xlabel("words"); a.legend(); a.grid(alpha=.3)

    plt.tight_layout()
    plt.savefig(f"{DRIVE}/fig_quality_{A['size']}.png", dpi=110, bbox_inches="tight")
    plt.show()

fig_quality(analysis)

In [ ]:
# --- Figure 3: language-model behavior (perplexity có mốc neo, profile vị trí, Zipf) ---
import math
import matplotlib.pyplot as plt

def fig_lm_behavior(A):
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

    # (1) perplexity held-out neo giữa floor và ceiling lý thuyết:
    #     floor   = e^(final train loss), mức mà train loss hứa hẹn
    #     ceiling = vocab size, model đoán mò uniform trên mọi token
    a = ax[0]
    labels, vals, colors = [], [], []
    ceil = A.get("vocab_size", 12000)
    labels.append("uniform guess\n(ceiling)"); vals.append(ceil); colors.append("#e5e7eb")
    if A.get("final_loss"):
        floor = math.exp(A["final_loss"])
        labels.append("e^train loss\n(floor)"); vals.append(floor); colors.append("#bbf7d0")
    labels.append("SLM 30M\nheld-out"); vals.append(A.get("perplexity", float("nan"))); colors.append("#2563eb")
    a.bar(labels, vals, color=colors)
    a.set_yscale("log")
    for i, v in enumerate(vals): a.text(i, v, f"{v:,.1f}", ha="center", va="bottom")
    a.set_title("Perplexity, log scale (good = near floor, far below ceiling)")
    a.grid(axis="y", alpha=.3, which="both")

    # (2) mean cross-entropy theo vị trí TƯƠNG ĐỐI trong truyện (mở truyện thường dễ nhất)
    a = ax[1]
    pos = A.get("pos_loss") or []
    xs = [i for i, v in enumerate(pos) if v is not None]
    if xs:
        n = len(pos)
        a.plot([100 * (i + 0.5) / n for i in xs], [pos[i] for i in xs],
               color="#dc2626", marker="o", ms=4)
    a.set_title("Mean loss by story position (flat-ish = consistent quality)")
    a.set_xlabel("story position (%)"); a.set_ylabel("cross-entropy"); a.grid(alpha=.3)

    # (3) Zipf rank-frequency (log-log): output sinh ra nên bám sát đường thật
    a = ax[2]
    zg, zr = A.get("zipf_gen") or [], A.get("zipf_real") or []
    if zg: a.loglog(range(1, len(zg) + 1), zg, label="SLM 30M", color="#2563eb")
    if zr: a.loglog(range(1, len(zr) + 1), zr, label="real fables", color="#94a3b8")
    a.set_title("Zipf: token rank vs frequency")
    a.set_xlabel("rank"); a.set_ylabel("frequency")
    a.legend(); a.grid(alpha=.3, which="both")

    plt.tight_layout()
    plt.savefig(f"{DRIVE}/fig_lm_{A['size']}.png", dpi=110, bbox_inches="tight")
    plt.show()

fig_lm_behavior(analysis)

In [ ]:
# --- Bảng verdict tự động: giá trị vs target cho mọi metric, merge vào JSON ---
# Ngưỡng là heuristic cho CHÍNH setup này (30M tham số, vocab 12k, fable TF1).
# Cell tự tính lại mọi số dẫn xuất từ `analysis` nên không phụ thuộc việc các cell
# figure đã chạy hay chưa.
import json, math
import numpy as np

def verdict_table(A):
    rows = []
    def add(metric, value, target, ok, warn=False):
        rows.append((metric, value, target, "PASS" if ok else ("WARN" if warn else "FAIL")))

    # training loss: < 1.5 vùng target, 1.5-2.0 partially trained, > 2.0 undertrained
    fl = A["losses"][-1]
    add("final train loss", f"{fl:.3f}", "< 1.5 (1.5-2.0 = WARN)", fl < 1.5, fl < 2.0)

    # scaling-law fit: đường loss sau warmup phải gần tuyến tính trên log-log
    steps, losses = A["steps"], A["losses"]
    n0 = max(1, len(steps) // 10)
    xs, ys = np.log(steps[n0:]), np.log(losses[n0:])
    slope, intercept = np.polyfit(xs, ys, 1)
    pred = slope * xs + intercept
    r2 = float(1 - ((ys - pred) ** 2).sum() / max(1e-12, ((ys - ys.mean()) ** 2).sum()))
    add("scaling-law fit R^2", f"{r2:.3f} (exp {slope:.2f})", "> 0.95 (> 0.90 = WARN)",
        r2 > 0.95, r2 > 0.90)

    # perplexity held-out so với floor lý thuyết e^(final train loss)
    ppl, floor = A["perplexity"], math.exp(fl)
    ratio = ppl / floor
    add("held-out perplexity", f"{ppl:.2f} (floor {floor:.2f}, {ratio:.2f}x)",
        "<= 1.5x floor (<= 3x = WARN)", ratio <= 1.5, ratio <= 3.0)

    # gap độ đa dạng so với fable thật
    q = A["quality"]
    for k, name in [("distinct1", "Distinct-1"), ("distinct2", "Distinct-2")]:
        g = abs(q[k]["gen"] - q[k]["real"]) / max(1e-9, q[k]["real"])
        add(f"{name} gap vs real", f"{g:.0%}", "<= 15% (<= 30% = WARN)", g <= 0.15, g <= 0.30)
    gsb = abs(q["self_bleu"]["gen"] - q["self_bleu"]["real"])
    add("Self-BLEU abs gap", f"{gsb:.3f}", "<= 0.05 (<= 0.15 = WARN)", gsb <= 0.05, gsb <= 0.15)

    # dải readability cho truyện thiếu nhi
    fg = q["flesch"]["gen"]
    add("Flesch reading ease", f"{fg:.1f}", "80-100 (60-80 = WARN)",
        80 <= fg <= 100, 60 <= fg < 80)

    # độ chồng lấp phân bố độ dài với fable thật
    lg, lr = A["len_gen"], A["len_real"]
    bins = np.linspace(min(lg + lr), max(lg + lr), 16)
    hg, _ = np.histogram(lg, bins=bins); hr, _ = np.histogram(lr, bins=bins)
    ov = float(np.minimum(hg / max(1, hg.sum()), hr / max(1, hr.sum())).sum())
    add("length distribution overlap", f"{ov:.0%}", ">= 50% (>= 30% = WARN)", ov >= 0.5, ov >= 0.3)

    # tỷ lệ template "wise old owl" trong output (prior data thật ~28%; pha 1 sinh 90%)
    if A.get("owl_rate_gen") is not None:
        orate = A["owl_rate_gen"]
        add("owl template rate (gen)", f"{orate:.0%}", "<= 30% (<= 60% = WARN)",
            orate <= 0.30, orate <= 0.60)

    w = max(len(r[0]) for r in rows)
    print(f"{'metric'.ljust(w)}  {'value'.ljust(30)}  {'target'.ljust(28)}  verdict")
    print("-" * (w + 72))
    for m, v, t, s in rows:
        print(f"{m.ljust(w)}  {v.ljust(30)}  {t.ljust(28)}  {s}")

    A["powerlaw"] = {"exponent": float(slope), "r2": r2}
    A["len_overlap"] = ov
    A["verdict"] = [{"metric": m, "value": v, "target": t, "verdict": s} for m, v, t, s in rows]
    json.dump(A, open(f"{DRIVE}/analysis_{A['size']}.json", "w"))
    n_pass = sum(1 for r in rows if r[3] == "PASS")
    n_warn = sum(1 for r in rows if r[3] == "WARN")
    n_fail = len(rows) - n_pass - n_warn
    print(f"\ntổng kết: {n_pass} PASS | {n_warn} WARN | {n_fail} FAIL  "
          f"(ngưỡng là heuristic cho 30M / vocab 12k / TF1)")

verdict_table(analysis)

## Step 6 - Export sang Ollama (GGUF)

Convert model **30M** đã train sang **GGUF (q8)** và ghi `Modelfile` cho Ollama với `TEMPLATE` tái tạo đúng format lúc train (`prompt + newline + <|story|>`, không có chat/system markup). Hai fix cho model nhỏ dùng custom BPE: ghi lại tokenizer_class (trong `train`) + patch pre-tokenizer của llama.cpp. Output nằm trong thư mục Drive.

In [ ]:
# --- Export model 30M sang GGUF + Ollama Modelfile, lưu vào Drive ---
import os, re, subprocess
if not os.path.exists("llama.cpp"):
    subprocess.run("git clone -q https://github.com/ggerganov/llama.cpp && pip -q install -r llama.cpp/requirements.txt", shell=True)

def convert(src, dst):
    # llama.cpp từ chối pre-tokenizer BPE lạ (tokenizer tự train có hash chkhsh riêng,
    # và hash ĐỔI mỗi khi train lại tokenizer) -> nếu bị từ chối, tự đọc chkhsh từ log
    # lỗi, patch get_vocab_base_pre để map về "gpt-2", rồi convert lại. Fail-fast nếu
    # vẫn lỗi (không để file GGUF cũ đánh lừa là export đã thành công).
    cmd = f"python llama.cpp/convert_hf_to_gguf.py {src} --outfile {dst} --outtype q8_0"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        m = re.search(r"chkhsh:\s*([0-9a-f]{64})", r.stdout + r.stderr)
        assert m, "convert lỗi nhưng không thấy chkhsh trong log:\n" + r.stderr[-2000:]
        chk = m.group(1)
        bp = "llama.cpp/conversion/base.py"; L = open(bp).read().split("\n")
        if not any(chk in x for x in L):
            for i, ln in enumerate(L):
                if 'raise NotImplementedError("BPE pre-tokenizer was not recognized' in ln:
                    ind = ln[:len(ln) - len(ln.lstrip())]
                    L[i] = f'{ind}if chkhsh == "{chk}": return "gpt-2"\n{ln}'; break
            open(bp, "w").write("\n".join(L))
            print(f"đã patch pre-tokenizer chkhsh {chk[:12]}... -> gpt-2")
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    assert r.returncode == 0, "convert vẫn lỗi:\n" + r.stderr[-2000:]

TMPL = 'TEMPLATE """{{ .Prompt }}\\n<|story|>"""\n'
for s in ["30M-p2"]:                             # model pha 2 (artifact hậu tố p2, không đè Run 3)
    tag = f"slm-{s.lower()}"
    assert os.path.isdir(f"out/{s}"), f"thiếu out/{s} - hãy train trước"
    convert(f"out/{s}", f"{DRIVE}/{tag}.gguf")
    open(f"{DRIVE}/Modelfile-{s}", "w").write(
        f"FROM ./{tag}.gguf\n{TMPL}"
        'PARAMETER temperature 0.8\nPARAMETER top_p 0.9\nPARAMETER repeat_penalty 1.1\n'
        'PARAMETER stop "<|end|>"\nPARAMETER num_ctx 512\n')
    print(f"OK: {DRIVE}/{tag}.gguf")
print("đã export lên Drive:", sorted(os.listdir(DRIVE)))